In [ ]:
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Mini-projet 2 – Effacement de consommation

## Partie 3 : Régulation collective

### Nouveau problème d'optimisation

On considère $n_l = 2$ logements. La dynamique du logement $j$ est désormais couplée à celle du voisin via l'équation (9). Ce couplage est **affine** en $(P^j_i, T^j_i, T^k_i)$ : les coefficients $h_{jk}$ sont des paramètres fixes. Le problème reste donc un **LP** (programmation linéaire), exactement dans le cadre de la section 2.2.2.1 du cours (contraintes affines, problème (34)), résolu par l'algorithme du simplexe (Appendice C.3).

Les variables de décision sont
$$x = (P^1_0,\ldots,P^1_{N-1},\; P^2_0,\ldots,P^2_{N-1},\; T^1_0,\ldots,T^1_N,\; T^2_0,\ldots,T^2_N) \in \mathbb{R}^{2(2N+1)}.$$

L'objectif est la **facture totale** :
$$\min \; \Delta t \sum_{i=0}^{N-1} c_i \left(P^1_i + P^2_i\right)$$

Les contraintes d'égalité sont la dynamique (9) pour chaque logement et les conditions initiales $T^j_0 = T^j_m$ (chaque logement démarre à sa propre température minimale). Les contraintes d'inégalité sont les bornes $0 \leq P^j_i \leq P_M$ et les contraintes de confort $T^j_m \leq T^j_i \leq T_M$ pour $i \in I_{occ}$, avec $T^1_m = 18°C$ et $T^2_m = 20°C$.

### Avantages et inconvénients du pilotage centralisé

**Avantages.** L'optimiseur dispose de l'information complète sur les deux logements et peut exploiter le couplage thermique : quand un logement chauffe, il transfère de la chaleur à son voisin, ce que l'optimiseur utilise pour réduire la facture totale. L'optimum global sur l'ensemble du système est garanti.

**Inconvénients.** L'optimiseur doit connaître les paramètres thermiques et les préférences de confort de **chaque** logement. Cela soulève des questions de vie privée. De plus, le problème grossit linéairement avec $n_l$ : pour un grand immeuble, le nombre de variables et de contraintes devient important. La section 2.2.2.2 du cours (dualité) suggère une alternative : utiliser les **multiplicateurs de Lagrange** associés aux contraintes comme prix-signaux pour un pilotage décentralisé, où chaque logement optimise séparément en réponse à un signal tarifaire commun.

In [ ]:
# Paramètres (modifiables à la volée)
t0   = 23.0
dt   = 0.5
N    = 48

c_cr = 1.0
c_pl = 3/2

T_M   = 30.0
T_m1  = 18.0
T_m2  = 20.0
T_in1 = T_m1
T_in2 = T_m2

h    = 0.05
k    = 0.01
b    = 1/500
P_M  = 5000.0
h12  = h

t = np.array([(t0 + i * dt) % 24 for i in range(N + 1)])
T_ext = 4 + 8 * np.exp(-(t - 12)**2 / 40)

def is_heure_creuse(heure):
    return (0 <= heure < 6) or (12 <= heure < 14)

def is_occupe(heure):
    return (7 <= heure <= 9) or (18 <= heure < 23)

c_tarif = np.array([c_cr if is_heure_creuse(t[i]) else c_pl for i in range(N + 1)])
I_occ   = [i for i in range(N + 1) if is_occupe(t[i])]

In [ ]:
def solve_collectif(c_pl_val):
    c_tarif_loc = np.array([c_cr if is_heure_creuse(t[i]) else c_pl_val for i in range(N + 1)])

    P1 = ca.SX.sym('P1', N)
    P2 = ca.SX.sym('P2', N)
    T1 = ca.SX.sym('T1', N + 1)
    T2 = ca.SX.sym('T2', N + 1)

    x = ca.vertcat(P1, P2, T1, T2)

    obj = dt * ca.dot(c_tarif_loc[:N], P1 + P2)

    # Dynamique couplée (équation 9) : coefficient global k + h + h12
    kh    = k + h + h12
    alpha = np.exp(-kh * dt)
    beta  = (1 - alpha) / kh

    g   = []
    lbg = []
    ubg = []

    g.append(T1[0]);  lbg.append(T_in1);  ubg.append(T_in1)
    g.append(T2[0]);  lbg.append(T_in2);  ubg.append(T_in2)

    for i in range(N):
        g.append(T1[i+1] - alpha * T1[i] - beta * b * P1[i] - beta * h12 * T2[i])
        lbg.append(beta * h * T_ext[i])
        ubg.append(beta * h * T_ext[i])

        g.append(T2[i+1] - alpha * T2[i] - beta * b * P2[i] - beta * h12 * T1[i])
        lbg.append(beta * h * T_ext[i])
        ubg.append(beta * h * T_ext[i])

    for i in I_occ:
        g.append(T1[i]);  lbg.append(T_m1);  ubg.append(T_M)
        g.append(T2[i]);  lbg.append(T_m2);  ubg.append(T_M)

    g = ca.vertcat(*g)

    lp = {'x': x, 'f': obj, 'g': g}
    solver = ca.qpsol('solver', 'highs', lp)

    lbx = [0.0]*N + [0.0]*N + [-ca.inf]*(N+1) + [-ca.inf]*(N+1)
    ubx = [P_M]*N + [P_M]*N + [ca.inf]*(N+1)  + [ca.inf]*(N+1)

    sol = solver(lbx=lbx, ubx=ubx, lbg=lbg, ubg=ubg)

    P1_sol = np.array(sol['x'][:N]).flatten()
    P2_sol = np.array(sol['x'][N:2*N]).flatten()
    T1_sol = np.array(sol['x'][2*N:2*N+N+1]).flatten()
    T2_sol = np.array(sol['x'][2*N+N+1:]).flatten()
    facture = float(sol['f'])

    return P1_sol, P2_sol, T1_sol, T2_sol, facture

In [ ]:
P1_opt, P2_opt, T1_opt, T2_opt, facture = solve_collectif(c_pl)
print(f"Facture totale optimale : {facture:.2f}")
print(f"T1 : min = {T1_opt.min():.2f} °C, max = {T1_opt.max():.2f} °C")
print(f"T2 : min = {T2_opt.min():.2f} °C, max = {T2_opt.max():.2f} °C")

In [ ]:
def plot_resultats(P1_sol, P2_sol, T1_sol, T2_sol, c_pl_val):
    c_tarif_loc = np.array([c_cr if is_heure_creuse(t[i]) else c_pl_val for i in range(N + 1)])
    P1_plot = np.append(P1_sol, 0.0)
    P2_plot = np.append(P2_sol, 0.0)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f'Régulation collective – c_pl = {c_pl_val:.2f}', fontsize=11)

    for ax in axes.flat:
        for ts, te in [(7, 9), (18, 23)]:
            ax.axvspan(ts, te, alpha=0.10, color='green')
        ax.set_xticks(range(0, 25, 2))
        ax.set_xlim(0, 24)
        ax.set_xlabel('Heure')

    occ_patch = mpatches.Patch(color='green', alpha=0.3, label="Occupation")

    for idx, (T_sol, P_plot, T_m, label) in enumerate([
        (T1_sol, P1_plot, T_m1, 'Logement 1'),
        (T2_sol, P2_plot, T_m2, 'Logement 2')
    ]):
        ax = axes[0, idx]
        ax.plot(t, T_sol, 'b-o', markersize=3, label=f'T {label} (°C)')
        ax.plot(t, T_ext, 'k--', linewidth=1, alpha=0.5, label='T extérieure')
        for ts, te in [(7, 9), (18, 23)]:
            ax.hlines(T_m, ts, te, colors='green', linestyles=':', linewidth=1.5)
            ax.hlines(T_M, ts, te, colors='red',   linestyles=':', linewidth=1.5)
        ax.set_ylabel('Température (°C)', color='b')
        ax.tick_params(axis='y', labelcolor='b')
        ax2 = ax.twinx()
        ax2.step(t, P_plot / P_M, 'r-', where='post', linewidth=1.5, alpha=0.7, label='P/P_M')
        ax2.set_ylabel('P / P_M', color='r')
        ax2.tick_params(axis='y', labelcolor='r')
        ax2.set_ylim(0, 1.3)
        lines1, _ = ax.get_legend_handles_labels()
        lines2, _ = ax2.get_legend_handles_labels()
        Tm_line = plt.Line2D([0], [0], color='green', linestyle=':', label=f'T_m = {T_m}°C')
        TM_line = plt.Line2D([0], [0], color='red',   linestyle=':', label=f'T_M = {T_M}°C')
        ax.legend(handles=lines1 + lines2 + [occ_patch, Tm_line, TM_line], fontsize=8, loc='upper left')
        ax.set_title(f'{label} – température et puissance')

        ax = axes[1, idx]
        ax.step(t, P_plot / P_M, 'b-', where='post', linewidth=2, label='P/P_M')
        ax.set_ylabel('P / P_M', color='b')
        ax.tick_params(axis='y', labelcolor='b')
        ax.set_ylim(0, 1.3)
        ax2 = ax.twinx()
        ax2.step(t, c_tarif_loc, 'r--', where='post', linewidth=1.5, alpha=0.8, label='Tarif')
        ax2.set_ylabel('Tarif', color='r')
        ax2.tick_params(axis='y', labelcolor='r')
        ax2.set_ylim(0, 2.5)
        lines1, _ = ax.get_legend_handles_labels()
        lines2, _ = ax2.get_legend_handles_labels()
        ax.legend(handles=lines1 + lines2, fontsize=8, loc='upper right')
        ax.set_title(f'{label} – puissance vs tarif')

    plt.tight_layout()
    plt.show()

plot_resultats(P1_opt, P2_opt, T1_opt, T2_opt, c_pl)

**Commentaire.** On retrouve pour les deux logements la stratégie de préchauffage en heures creuses. Le logement 2 ($T^2_m = 20°C$) requiert un préchauffage plus important que le logement 1 ($T^1_m = 18°C$) pour satisfaire sa contrainte de confort plus stricte. Le couplage thermique entre les deux logements joue un rôle : quand le logement 2 chauffe intensément, il transfère de la chaleur vers le logement 1, ce qui permet à ce dernier de réduire légèrement sa propre puissance. L'optimiseur centralisé exploite ce transfert pour minimiser la facture totale.

In [ ]:
# Effet incitatif du tarif en régulation collective
c_pl_values = [3/2, 7/4, 2]
colors = ['steelblue', 'darkorange', 'crimson']

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for c_pl_v, col in zip(c_pl_values, colors):
    P1_v, P2_v, _, _, facture_v = solve_collectif(c_pl_v)
    axes[0].step(t, np.append(P1_v, 0.0) / P_M, where='post', color=col, linewidth=1.5,
                 label=f'c_pl={c_pl_v:.2f} | facture={facture_v:.0f}')
    axes[1].step(t, np.append(P2_v, 0.0) / P_M, where='post', color=col, linewidth=1.5,
                 label=f'c_pl={c_pl_v:.2f}')

for ax in axes:
    for ts, te in [(7, 9), (18, 23)]:
        ax.axvspan(ts, te, alpha=0.1, color='green')
    ax.set_xticks(range(0, 25, 2))
    ax.set_xlim(0, 24)
    ax.legend(fontsize=9)

axes[0].set_ylabel('P1 / P_M')
axes[1].set_ylabel('P2 / P_M')
axes[1].set_xlabel('Heure')
axes[0].set_title('Logement 1 – effet incitatif du tarif')
axes[1].set_title('Logement 2 – effet incitatif du tarif')

plt.tight_layout()
plt.show()

**Commentaire.** L'effet incitatif est plus marqué pour le logement 1 que pour le logement 2. Le logement 2 est contraint à $T^2_m = 20°C$ : sa contrainte de confort est active quelle que soit la valeur de $c_{pl}$, ce qui bride sa flexibilité. Au sens des conditions KKT (section 2.2.2.1 du cours), le multiplicateur de Lagrange associé à cette contrainte reste strictement positif — la contrainte est active et l'optimiseur ne peut pas s'en écarter. Le logement 1, avec $T^1_m = 18°C$ moins contraignant, dispose d'une plus grande liberté pour reporter sa consommation vers les heures creuses lorsque $c_{pl}$ augmente.